# 实践项目 05：空间转录组表达超分辨率

本 Notebook 使用课程准备的配对 H&E、LR、HR 和 split 数据。LR 表示 16 μm Snap25 输入，HR 表示 2 μm 参考表达图，模型学习在细网格上估计局部表达。

Kaggle 是本项目的首选实践入口。打开公开 Notebook 后，点击“复制并编辑”保存到自己的账户，再按单元格顺序运行。下载 Notebook 到电脑运行是补充方式。

代码中用整行注释标出了需要填写的位置。先阅读当前单元格的输入、处理和输出，再修改标记区域。合理利用 AI 工具理解问题、学习知识并尝试给出适当的解决方案。

## 任务总览

1. 核对四个字段的 shape、split 数量和表达范围。
2. 查看同一区域的 H&E、LR 和 HR。
3. 补全 H&E 与粗尺度表达联合输入的轻量网络。
4. 完成 log 空间损失和 8×8 聚合一致性。
5. 比较模型与插值基线的 MAE、相关性、聚合误差和图像结果。

## 需要保存的结果

`task5_data_visualization.png`、`task5_training_curve.png`、`task5_prediction_visualization.png`、`task5_result.json`。

In [ ]:
from pathlib import Path
import json, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT = Path('/kaggle/working'); OUT.mkdir(exist_ok=True)
DATA_PATH = None
candidates = sorted(Path('/kaggle/input').rglob('kydw-try-a05-paired-patches.npz'))
if DATA_PATH is None and candidates: DATA_PATH = candidates[0]
assert DATA_PATH is not None, '请挂载包含 kydw-try-a05-paired-patches.npz 的课程数据集。'
data = np.load(DATA_PATH, allow_pickle=True)
required = {'he','lr','hr','split'}
assert required.issubset(data.files), required - set(data.files)
he = data['he'].astype(np.float32) / 255.0; lr = data['lr'].astype(np.float32); hr = data['hr'].astype(np.float32); split = data['split'].astype(str)
print('he/lr/hr:', he.shape, lr.shape, hr.shape, 'splits:', {k: int((split == k).sum()) for k in np.unique(split)})

## 任务 1：核对输入字段与空间划分

H&E、LR 和 HR 必须覆盖同一空间区域。LR 在每个 8×8 区域内记录粗尺度总量。

In [ ]:
# ===== 项目05·任务1·学生填写区（开始） =====
# TODO：统计各 split 数量、shape、非零比例和最大值。
summary = None
# ===== 项目05·任务1·学生填写区（结束） =====
print(summary)

## 任务 2：补全融合网络

输入为 4 个通道，输出为 1 个通道的细尺度表达预测。

In [ ]:
class STDataset(Dataset):
    def __init__(self, kind): self.ids = np.where(split == kind)[0]
    def __len__(self): return len(self.ids)
    def __getitem__(self, k):
        i = int(self.ids[k]); lr_total = lr[i]
        x = np.concatenate([he[i], np.log1p(lr_total / 64.0)], axis=0)
        y = np.log1p(hr[i])
        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(lr_total), i

class SRNet(nn.Module):
    def __init__(self):
        super().__init__()
        # ===== 项目05·任务2·学生填写区（开始） =====
        # TODO：补全 4 通道输入、1 通道输出的轻量网络。
        self.body = None
        # ===== 项目05·任务2·学生填写区（结束） =====
    def forward(self, x): return torch.nn.functional.softplus(x[:, 3:4] + self.body(x))

model = SRNet().to(DEVICE)
print(model)

## 任务 3：损失和聚合一致性

预测值按 8×8 区域求和后，应与 LR 中的粗尺度总量接近。

In [ ]:
def aggregate8(x): return torch.nn.functional.avg_pool2d(x, 8, 8) * 64

def loss_fn(pred_log, target_log, lr_raw):
    pred = torch.expm1(pred_log).clamp_min(0)
    target = torch.expm1(target_log).clamp_min(0)
    l1 = (pred_log - target_log).abs().mean()
    # ===== 项目05·任务3·学生填写区（开始） =====
    # TODO：比较 aggregate8(pred) 与 avg_pool2d(lr_raw, 8, 8)。
    consistency = None
    # ===== 项目05·任务3·学生填写区（结束） =====
    return l1 + .1 * consistency

## 任务 4：训练与结果比较

完成训练后，比较模型和插值基线的 MAE、相关性和预测图。

In [ ]:
# ===== 项目05·任务4·学生填写区（开始） =====
# TODO：完成训练、验证选模、模型与插值基线比较，以及结果保存。
# ===== 项目05·任务4·学生填写区（结束） =====
print('输出文件应写入', OUT)